# Custom GPT Training 03: SFT 指令微调 教案

**课程名称：** SFT 指令微调：从预训练模型到对话模型

**预计总时长：** 约 80 分钟

**源文件：** `Custom_GPT_Training/03_SFT_Training.ipynb`（共 25 个 Cell，Cell 0-24）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 环境准备 + 全局概览 | Cell 0-3 | 8 min |
| 8-18 min | SFT 训练数据 + ChatML 格式化 | Cell 4-7 | 10 min |
| 18-32 min | SFT Dataset（Loss Masking 实现） | Cell 8-11 | 14 min |
| 32-40 min | **休息 + 回顾** | -- | 8 min |
| 40-55 min | SFT Trainer + 执行训练 + Loss 可视化 | Cell 12-17 | 15 min |
| 55-70 min | 测试 SFT 模型 + Base vs SFT 对比 | Cell 18-23 | 15 min |
| 70-80 min | 总结 + 下一章预告 | Cell 24 | 10 min |

---

## 课前检查清单

- [ ] GPU 环境就绪（CPU 可用但训练较慢）
- [ ] `torch`, `matplotlib`, `tqdm`, `numpy` 已安装
- [ ] 预训练模型已完成（`models/custom_gpt/pretrained_model` 目录存在）
- [ ] Tokenizer 已保存（`models/custom_gpt/tokenizer.pkl` 存在）
- [ ] SFT 数据文件就绪：`data/custom_sft_train.jsonl`、`data/custom_sft_val.jsonl`、`data/custom_sft_test.jsonl`
- [ ] `custom_gpt.py` 模块可导入（CustomGPT, GPTConfig, SimpleTokenizer, count_parameters）
- [ ] 中文字体可用（matplotlib 绘图用）

---

## 第一段：开场与环境准备（Cell 0-3）

📍 运行 Cell 0-1（Markdown 导读 + 学习路线），运行 Cell 3（import + 环境配置 + 设备检测）

⏱ 时间分配：8 分钟（开场动机 3 分钟 + 环境运行 2 分钟 + 路线定位 3 分钟）

🎯 本段目标
- 建立学习动机：预训练之后为什么还需要 SFT？
- 理解预训练 vs SFT 的本质区别（续写 vs 回答）
- 确认环境就绪（GPU/CPU、依赖库、预训练模型）

🗣 讲课话术

> 大家好！上一节课我们完成了预训练，得到了一个 14.31M 参数的自建 GPT 模型。它已经学会了中文语言的规律——能续写文本。但是，大家试试看，你对它说"什么是机器学习？"，它会怎么样？
>
> 看 Cell 0 的对比：
> ```
> 预训练: 输入"机器学习是人工智能" -> 预测下一个token（续写）
> SFT:    输入"<|user|>什么是机器学习？" -> 生成高质量回答
> ```
>
> 预训练模型就像一个**读了很多书但从没跟人说过话的学者**。你问他问题，他不是回答你，而是接着你的话继续往下"背书"。SFT 就是教他"别人问你问题时，你应该怎么回答"。
>
> Cell 0 列出了 SFT 的四个核心技术点：ChatML 格式、Loss Masking、指令多样性、数据质量。今天我们每一个都会亲手实现。
>
> 看 Cell 1 的路线图——我们处于 Part 3，承接 Part 2 预训练，后面还有 Part 4 DPO 对齐。SFT 是中间最关键的一步：把"能力"变成"行为"。
>
> 现在运行 Cell 3。（运行 Cell 3）看到输出了吗？"使用设备: cuda"（或 cpu）。确认设备没问题，依赖库全部导入成功。

👀 输出要点
- 设备信息：`使用设备: cuda`（或 `cpu`）
- 各依赖库成功导入无报错
- 随机种子已设置（torch.manual_seed(42)）

❓ 预判 Q&A
- **Q：CPU 能跑吗？**
  A：能跑，Cell 0 说了"CPU 可跑（训练时间更长），GPU 可加速"。CPU 上训练大概慢 3-5 倍，但不影响学习。
- **Q：如果 Part 2 的预训练模型没保存怎么办？**
  A：Cell 15 有兜底逻辑——如果预训练模型不存在，会创建一个新的随机初始化模型。效果会差一些，但流程能跑完。
- **Q：SFT 和 Fine-tuning 有什么区别？**
  A：Fine-tuning 是泛称；SFT 特指用"指令-回复"对来微调，目标是让模型学会遵循指令格式。

➡️ 转场：环境没问题。接下来我们看 SFT 的数据长什么样，以及 ChatML 格式是怎么回事。

---

## 第二段：SFT 训练数据 + ChatML 格式化（Cell 4-7）

📍 浏览 Cell 4（数据说明 Markdown），运行 Cell 5（加载数据），浏览 Cell 6（ChatML Markdown），运行 Cell 7（ChatMLFormatter 定义 + 示例输出）

⏱ 时间分配：10 分钟（数据概览 4 分钟 + ChatML 格式 6 分钟）

🎯 本段目标
- 了解 SFT 数据的结构：instruction + response + category
- 掌握 ChatML 格式的设计哲学和角色标记
- 理解 format vs format_prompt_only 的区别（训练 vs 推理）

🗣 讲课话术

> 运行 Cell 5。（运行 Cell 5）看输出：
> ```
> 训练集: 8000 条 | 验证集: 1000 条 | 测试集: 1000 条
> 任务类型: ['知识问答']
> ```
>
> 我们有 8000 条训练数据，全部是"知识问答"类型。看示例：
> ```
> instruction: '用一句话解释：深度学习是什么？'
> response: '深度学习是用多层神经网络学习数据表示的方法。'
> ```
>
> 注意数据特点——Cell 4 说了，这是"短知识问答"，回答都是**简洁、单句的事实性回答**。这是刻意设计的，方便我们观察模型是否真的学会了指令理解。
>
> 接下来看 ChatML 格式。运行 Cell 7。（运行 Cell 7）看输出：
> ```
> <|system|>你是一个有帮助的AI助手。<|endoftext|>
> <|user|>用一句话解释：深度学习是什么？<|endoftext|>
> <|assistant|>深度学习是用多层神经网络学习数据表示的方法。<|endoftext|>
> ```
>
> 三个角色标记：`<|system|>`、`<|user|>`、`<|assistant|>`，加上结束标记 `<|endoftext|>`。
>
> 为什么需要这些标记？想象你在看一个**剧本**。如果没有角色名，你根本分不清谁在说话、什么时候该停。ChatML 就是这个"剧本格式"——它告诉模型：这是系统设定、这是用户在说话、这是你该回复的地方、这里该停了。
>
> 注意 `ChatMLFormatter` 有两个方法：`format()` 用于训练（包含完整对话），`format_prompt_only()` 用于推理（只到 `<|assistant|>` 就停，等模型续写）。这个区别很重要，后面训练和测试都要用到。

👀 输出要点
- 数据规模：训练 8000 / 验证 1000 / 测试 1000
- 任务类型：知识问答（短句事实性回答）
- ChatML 格式：`<|system|>` + `<|user|>` + `<|assistant|>` + `<|endoftext|>`
- 示例数据：instruction='用一句话解释：深度学习是什么？' -> response='深度学习是用多层神经网络学习数据表示的方法。'

❓ 预判 Q&A
- **Q：ChatML 是唯一的对话格式吗？**
  A：不是。Llama 用 `[INST]` 格式，Alpaca 用 `### Instruction:` 格式。ChatML 是 OpenAI 推广的标准，也是目前最常用的。
- **Q：system prompt 一定需要吗？**
  A：不一定。`format()` 方法有 `include_system` 参数可以关掉。但 system prompt 能帮助模型理解自己的角色定位。
- **Q：8000 条数据够吗？**
  A：对于我们的小模型和简单任务来说足够了。工业界的 SFT 数据通常是 10 万到百万级别。

➡️ 转场：数据和格式都清楚了。接下来是 SFT 最核心的技术点——Loss Masking。这决定了模型到底学什么、不学什么。

---

## 第三段：SFT Dataset + Loss Masking 实现（Cell 8-11）

📍 浏览 Cell 8（Loss Masking 说明 Markdown），运行 Cell 9（SFTDataset 类定义），运行 Cell 10（加载 tokenizer + 创建数据集 + DataLoader），运行 Cell 11（Loss Masking 可视化）

⏱ 时间分配：14 分钟（Dataset 构造 6 分钟 + 数据加载 3 分钟 + Mask 可视化 5 分钟）

🎯 本段目标
- 理解 Loss Masking 的原理：只对 assistant 回答部分计算 loss
- 掌握 SFTDataset 的构建流程：ChatML 编码 -> 定位 assistant 起始位置 -> labels 设置
- 通过可视化直观看到哪些 token 参与训练、哪些被忽略

🗣 讲课话术

> 这一段是整节课最重要的部分。先看 Cell 8 的标题——"SFT数据集（带Loss Masking）"。
>
> 什么是 Loss Masking？我用一个考试的比喻：你在改卷子的时候，**只给答案评分，不给题目评分**。用户的指令是"题目"，模型的回复是"答案"。如果把题目也算进 loss，模型会花精力去"背题目"而不是"答好题"。
>
> 技术上怎么实现？看 Cell 9 的 SFTDataset 类。核心逻辑在 `__getitem__` 方法里：
> 1. 用 `ChatMLFormatter.format()` 把 instruction + response 变成完整 ChatML 文本
> 2. 用 `ChatMLFormatter.format_prompt_only()` 只编码到 `<|assistant|>`，得到 prompt 的长度——这就是 `assistant_start` 位置
> 3. 创建 labels：`assistant_start` 之前全部设为 **-100**，之后才是真正的 token id
> 4. PyTorch 的 CrossEntropyLoss 默认 `ignore_index=-100`，会自动忽略这些位置
>
> 运行 Cell 10。（运行 Cell 10）看输出：
> ```
> 加载tokenizer, 词表大小: 381
> SFT数据集: 8000 条
> 最大长度: 256
> ```
>
> 词表只有 381 个 token——因为我们用的是字符级分词器，一个汉字就是一个 token。MAX_LENGTH=256，BATCH_SIZE=8。
>
> 现在运行 Cell 11 看 Loss Masking 的可视化。（运行 Cell 11）这个输出非常直观：
> ```
> Token ID | Label | 是否计算Loss
>      1  |  -100 | x  '<BOS>'
>    370  |  -100 | x  '<'
>    245  |  -100 | x  '|'
>    ...
> ```
>
> 前面全是 x（不计算 loss）——包括 `<BOS>`、`<|system|>`、系统提示、`<|user|>`、用户指令这些。一直到 `<|assistant|>` 之后才变成有效标签。
>
> 看最后的统计：**总 token 数 255，计算 loss 的 token 数只有 35**。也就是说，只有约 14% 的 token 参与训练！剩下 86% 都是"题目"部分，被 mask 掉了。SFT 的精华就在这 14% 的 assistant 回复里。

👀 输出要点
- Tokenizer 词表大小：381（字符级）
- 数据集：训练 8000 样本、验证 1000 样本
- MAX_LENGTH: 256, BATCH_SIZE: 8
- Loss Masking 可视化：x 表示 label=-100（不参与），有效标签表示参与训练
- 关键数据：总 255 token，只有 35 个参与 loss 计算（约 14%）

❓ 预判 Q&A
- **Q：为什么 -100？不能用 0 或其他值吗？**
  A：-100 是 PyTorch CrossEntropyLoss 的默认 `ignore_index`。你也可以用其他值，但需要手动设置 `nn.CrossEntropyLoss(ignore_index=你的值)`。
- **Q：input_ids 和 labels 为什么要错位一位（[:-1] 和 [1:]）？**
  A：这是自回归语言模型的标准做法。输入是 token 1 到 N-1，标签是 token 2 到 N——模型在每个位置预测"下一个"token。
- **Q：padding 部分的 labels 也是 -100 吗？**
  A：是的！看代码 `labels = labels + [-100] * pad_len`，padding 也不应该参与 loss 计算。
- **Q：attention_mask 是做什么的？**
  A：告诉模型哪些位置是真实 token（1），哪些是 padding（0）。防止模型"关注"无意义的填充位置。

➡️ 转场：Loss Masking 是 SFT 最核心的技术点，大家一定要理解透。接下来我们休息一下，回顾前半段内容。

---

## 休息 + 回顾（第 32-40 分钟）

⏱ 时间分配：8 分钟（休息 5 分钟 + 回顾 3 分钟）

**三句话回顾前半段：**

1. **SFT 的本质**是把预训练模型从"续写机器"变成"对话助手"。预训练教会模型语言能力（知识），SFT 教会模型对话行为（格式）。ChatML 用 `<|system|>`、`<|user|>`、`<|assistant|>`、`<|endoftext|>` 四个标记定义对话结构。

2. **Loss Masking 是 SFT 的核心技术**。只对 assistant 回复部分计算 loss，user 指令和 system 提示的 labels 设为 -100。在我们的数据中，255 个 token 里只有 35 个参与训练（约 14%），精华就在这一小部分里。

3. **数据规模**：8000 条训练 / 1000 条验证 / 1000 条测试，全部是知识问答。词表 381 个字符级 token，序列最大长度 256。模型参数 14.31M。

**下一段预告：** 我们要正式开始 SFT 训练了！看 loss 从 0.3582 降到 0.0011，然后对比 Base 模型和 SFT 模型的回答差距。

---

## 第四段：SFT Trainer + 执行训练 + Loss 可视化（Cell 12-17）

📍 浏览 Cell 12（Trainer 说明 Markdown），运行 Cell 13（SFTTrainer 类定义），运行 Cell 15（加载预训练模型），运行 Cell 16（创建 Trainer + 执行训练），运行 Cell 17（Loss 曲线可视化）

⏱ 时间分配：15 分钟（Trainer 设计 4 分钟 + 模型加载 2 分钟 + 训练执行 5 分钟 + Loss 分析 4 分钟）

🎯 本段目标
- 理解 SFT Trainer 与预训练 Trainer 的关键区别
- 理解 SFT 关键超参数的选择（学习率、epoch 数）
- 观察 loss 下降过程并分析收敛行为

🗣 讲课话术

> 先看 Cell 13 的 SFTTrainer 类。它和预训练 Trainer 有三个关键区别，Cell 12 的注释写得很清楚：
> 1. **使用带 masking 的 labels**——`compute_loss` 方法用 `nn.CrossEntropyLoss(ignore_index=-100)`
> 2. **较小的学习率**——默认 `lr=5e-5`，比预训练小一个数量级
> 3. **较少的训练轮次**——我们只训 5 个 epoch
>
> 为什么学习率要小？因为模型已经学好了语言能力，我们只是"微调"它的行为。就像调钢琴——你不会把琴弦全拧掉重来，只是微微拧几下。学习率太大会导致**灾难性遗忘**——模型把预训练学到的知识忘掉了。
>
> 还有两个技术细节：
> - **Warmup**：前 20 步学习率从 0 线性增长到 5e-5，然后余弦退火。这防止训练开始时的大梯度破坏模型。
> - **梯度裁剪** `max_grad_norm=1.0`：防止梯度爆炸。
>
> 运行 Cell 15 加载预训练模型。（运行 Cell 15）看到了：
> ```
> 加载预训练模型...
> 模型参数: 14.31M
> ```
>
> 14.31M 参数——这是从 Part 2 预训练好的模型。如果没有预训练模型，它会创建一个新的（d_model=384, n_heads=6, n_layers=6）。
>
> 现在运行 Cell 16 开始训练！（运行 Cell 16）
>
> 看训练参数：
> ```
> 模型参数: 14.31M
> 训练样本: 8000
> Epochs: 5
> ```
>
> 8000 条数据、batch_size=8，一个 epoch 有 1000 步，5 个 epoch 总共 5000 步。
>
> （训练过程中讲解）大家看 loss 变化——
> - **Epoch 1**：训练 loss 0.3582，验证 loss 0.0040。第一轮就已经学到很多了！
> - **Epoch 2**：训练 loss 降到 0.0039，验证 loss 0.0015。模型已经基本学会了。
> - **Epoch 5**：训练 loss 0.0011，验证 loss 0.0006。几乎完美。
>
> 注意看，每个 epoch 都"保存最佳模型"——因为验证 loss 在持续下降。这是 best checkpoint 策略：只保留验证集上表现最好的模型。
>
> 运行 Cell 17 看 Loss 曲线。（运行 Cell 17）左图是训练 loss——Epoch 1 快速下降后趋于平稳，后面 4 个 epoch 在低位微调。右图是验证 loss——5 个 epoch 持续下降，没有上升，说明**没有过拟合**。

👀 输出要点
- 模型参数：14.31M，从预训练模型初始化
- 训练配置：lr=5e-5, epochs=5, batch_size=8, warmup_steps=20
- Loss 变化：Epoch 1 训练 loss 0.3582 -> Epoch 5 训练 loss 0.0011
- 验证 Loss：0.0040 -> 0.0006（持续下降，无过拟合）
- 每个 epoch 都保存了最佳模型
- Loss 曲线：训练 loss 快速下降后趋稳，验证 loss 持续下降

❓ 预判 Q&A
- **Q：为什么 SFT 的初始 loss 只有 0.3582 而不是很大的数？**
  A：因为模型不是随机初始化的！它经过预训练，已经会说中文了。0.3582 意味着模型一开始就对正确 token 给了约 70% 的概率（e^(-0.3582) 约等于 0.70）。
- **Q：为什么训 5 个 epoch？会不会过拟合？**
  A：从验证 loss 看没有过拟合（持续下降）。实际工业界 SFT 通常训 1-3 个 epoch，因为数据量更大。我们的小数据集多训几轮没问题。
- **Q：warmup 20 步是怎么确定的？**
  A：经验值。通常设为总步数的 1-5%。我们总共 5000 步，20 步约 0.4%，属于保守设置。
- **Q：weight_decay=0.01 是做什么的？**
  A：L2 正则化，防止参数过大，有助于泛化。0.01 是 AdamW 的常用默认值。

➡️ 转场：训练完了！loss 降到了 0.0006。但 loss 低不代表模型真的会回答问题。接下来我们实际测试——让模型回答问题，看看效果如何。

---

## 第五段：测试 SFT 模型 + Base vs SFT 对比（Cell 18-23）

📍 浏览 Cell 18（测试说明 Markdown），运行 Cell 19（chat 函数定义），运行 Cell 20（加载 SFT 模型），运行 Cell 21（测试集问答测试），浏览 Cell 22（对比说明 Markdown），运行 Cell 23（Base vs SFT 对比）

⏱ 时间分配：15 分钟（chat 函数 3 分钟 + 问答测试 5 分钟 + Base vs SFT 对比 7 分钟）

🎯 本段目标
- 理解推理时 ChatML 格式的使用（format_prompt_only）
- 通过测试集验证 SFT 模型的回答质量
- 直观对比 Base 模型和 SFT 模型的巨大差距

🗣 讲课话术

> 先看 Cell 19 的 `chat` 函数。推理流程是：
> 1. 用 `format_prompt_only()` 把用户指令包装成 ChatML 格式（只到 `<|assistant|>`）
> 2. Tokenize 成 input_ids
> 3. 调用 `model.generate()` 让模型续写
> 4. 从输出中提取 `<|assistant|>` 到 `<|endoftext|>` 之间的内容
>
> 注意参数：`temperature=0.2`（接近贪心解码，输出更确定）、`do_sample=False`（不采样，直接取概率最大的 token）。
>
> 运行 Cell 20 加载 SFT 模型。运行 Cell 21 测试。（运行 Cell 20-21）
>
> 大家看结果——**5 个测试问题，模型回答全部与期望一致！**
> ```
> 指令: 用一句话解释：深度学习是什么？
> 期望: 深度学习是用多层神经网络学习数据表示的方法。
> 输出: 深度学习是用多层神经网络学习数据表示的方法。
> ```
>
> 机器学习、监督学习、无监督学习、强化学习——每一个都完美匹配。
>
> 但更震撼的是 Base vs SFT 的对比。运行 Cell 23。（运行 Cell 23）
>
> ```
> [Base模型] (直接续写，不理解指令):
>   用一解释深度学习是。征程要置学制应应程要置证与源
> [SFT模型] (理解指令，生成回答):
>   深度学习是用多层神经网络学习数据表示的方法。
> ```
>
> 看到区别了吗？**Base 模型的输出完全是胡言乱语**——"用一解释深度学习是。征程要置学制应应程要置证与源"。它不理解指令，只是在做无意义的续写。
>
> 而 **SFT 模型的回答精准、简洁、正确**。同一个 14.31M 参数的模型，知识没有变化，只是通过 SFT 学会了"怎么回答问题"。
>
> 这就是 SFT 的价值：**不教新知识，只教对话格式和行为模式**。

👀 输出要点
- chat 函数：format_prompt_only -> generate -> 提取 assistant 回复
- 测试集 5 个问题全部回答正确，输出与期望完全一致
- Base 模型输出："用一解释深度学习是。征程要置学制应应程要置证与源"（胡言乱语）
- SFT 模型输出："深度学习是用多层神经网络学习数据表示的方法。"（精准正确）
- 核心洞察：同一模型，SFT 前后天壤之别

❓ 预判 Q&A
- **Q：为什么 Base 模型输出这么差？**
  A：因为 Base 模型从来没见过 ChatML 格式。它看到 `<|user|>` 这些标记不知道是什么意思，就当普通文本胡乱续写。而且字符级分词器的词表很小（381），Base 模型的语言能力本身就有限。
- **Q：5 个问题都对了，是不是测试集太简单？**
  A：确实是相对简单的任务（短知识问答），但重点不是准确率本身，而是对比——同一个模型，SFT 前后从胡言乱语到精准回答，这个变化才是要学的。
- **Q：temperature 设为 0.2 和 1.0 有什么区别？**
  A：temperature 越低输出越确定（贪心），越高输出越随机（多样）。测试时用低 temperature 看模型的"最佳答案"，实际使用时可以调高一点增加多样性。
- **Q：SFT 模型会不会只是"背答案"？**
  A：好问题！这 5 个测试样本和训练集确实可能有重叠的问题模式。真正的泛化测试需要用模型没见过的指令。这也是为什么后面还需要 DPO 进一步优化。

➡️ 转场：对比效果非常明显。最后我们总结全课，并预告下一步——DPO 训练。

---

## 第六段：总结 + 下一章预告（Cell 24）

📍 浏览 Cell 24（总结 Markdown）

⏱ 时间分配：10 分钟（总结回顾 5 分钟 + 练习 3 分钟 + 预告 2 分钟）

🎯 本段目标
- 串联全课知识点，形成完整认知框架
- 通过快速练习巩固核心概念
- 预告 DPO 训练

🗣 讲课话术

> 看 Cell 24 的总结。这节课完成了三件事：
>
> **第一，ChatML 格式化。** 标准化的对话模板，三个角色标记 `<|system|>`、`<|user|>`、`<|assistant|>` 加结束标记 `<|endoftext|>`。训练用 `format()`（包含完整对话），推理用 `format_prompt_only()`（等模型续写）。
>
> **第二，Loss Masking。** SFT 最核心的技术点。只对 assistant 回答部分计算 loss，其他部分 labels 设为 -100。在我们的数据中，约 14% 的 token 参与训练。用 PyTorch 的 `CrossEntropyLoss(ignore_index=-100)` 自动实现。
>
> **第三，SFT 训练。** 从预训练模型初始化，较小学习率（5e-5），5 个 epoch。训练 loss 从 0.3582 降到 0.0011，验证 loss 从 0.0040 降到 0.0006。
>
> 效果对比：
> - **Base 模型**："用一解释深度学习是。征程要置学制应应程要置证与源"（胡言乱语）
> - **SFT 模型**："深度学习是用多层神经网络学习数据表示的方法。"（精准回答）
>
> Cell 24 有一句精炼的总结："之前只会续写文本，不理解指令；之后理解用户意图，生成针对性回答。"
>
> 快速练习——大家回答三个问题：
> 1. Loss Masking 中，labels 设为多少的位置会被忽略？（答：-100）
> 2. SFT 的学习率为什么比预训练小？（答：避免破坏预训练学到的知识 / 灾难性遗忘）
> 3. ChatML 中 `format_prompt_only()` 用在什么场景？（答：推理 / 测试时，等模型续写回答）
>
> **下一步预告：** Part 4 是 DPO 训练。SFT 后模型会回答了，但回答的"好不好"是另一回事。DPO（Direct Preference Optimization）让模型学习人类偏好——给定同一个问题的两个回答，学会选择更好的那个。这是从"会回答"到"回答得好"的关键一步。

👀 输出要点
- 完整管线：预训练模型 -> ChatML 格式 -> Loss Masking -> SFT 训练 -> 对话模型
- 三大核心收获：ChatML 格式化、Loss Masking、SFT 训练配置
- SFT 前后效果：胡言乱语 -> 精准回答

❓ 预判 Q&A
- **Q：SFT 之后还需要做什么才能变成 ChatGPT？**
  A：还需要 RLHF 或 DPO（Part 4 会讲）——让模型不仅会回答，还要回答得"好"、符合人类偏好。完整管线是：预训练 -> SFT -> RLHF/DPO。
- **Q：本课的 SFT 和工业界有什么差距？**
  A：原理完全一致。差距在规模——工业界用更大的模型（数十亿参数）、更多的数据（数十万条）、更复杂的数据清洗和质量控制流程。
- **Q：我能用这个流程训练自己的对话模型吗？**
  A：完全可以！准备好你的 instruction-response 数据对，替换 SFT_DATA，其他流程完全一样。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，环境准备 | 0-3 |
| 8 | SFT 数据 + ChatML 格式化 | 4-7 |
| 18 | SFT Dataset + Loss Masking | 8-11 |
| 32 | **休息 + 回顾** | -- |
| 40 | SFT Trainer + 训练执行 + Loss 可视化 | 12-17 |
| 55 | 测试 SFT 模型 + Base vs SFT 对比 | 18-23 |
| 70 | 总结 + 下一章预告 | 24 |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\mathcal{L}_{\text{SFT}}(\theta) = -\sum_{t \in \text{assistant}} \log p_{\theta}(y_t \mid y_{<t}, x)$$

其中 $x$ 为 system + user 输入（labels=-100，不参与 loss），$y$ 为 assistant 回复（参与 loss）。

### 模型配置

| 参数 | 值 |
|:---|:---|
| 基座模型 | CustomGPT（Part 2 预训练） |
| 模型参数量 | 14.31M |
| d_model | 384 |
| n_heads | 6 |
| n_layers | 6 |
| d_ff | 1536 |
| 词表大小 | 381（字符级分词器） |

### 数据规模速查

| 指标 | 值 |
|:---|:---|
| 训练集 | 8,000 条 |
| 验证集 | 1,000 条 |
| 测试集 | 1,000 条 |
| 任务类型 | 知识问答 |
| MAX_LENGTH | 256 |
| 总 token 数（示例） | 255 |
| 参与 loss 的 token 数（示例） | 35（约 14%） |

### 训练超参数

| 参数 | 值 |
|:---|:---|
| 学习率（LR） | 5e-05 |
| Epochs | 5 |
| Steps/Epoch | 1,000 |
| 总步数 | 5,000 |
| Batch Size | 8 |
| 优化器 | AdamW (weight_decay=0.01) |
| Warmup Steps | 20 |
| 梯度裁剪 | 1.0 |
| LR 调度 | Linear warmup + Cosine decay |

### 训练结果速查

| 指标 | 值 |
|:---|:---|
| Epoch 1 训练 Loss | 0.3582 |
| Epoch 1 验证 Loss | 0.0040 |
| Epoch 5 训练 Loss | 0.0011 |
| Epoch 5 验证 Loss | 0.0006 |
| Base 模型输出 | 胡言乱语（不理解指令） |
| SFT 模型输出 | 精准回答（与期望一致） |

---

## 附录 C：应急预案

### 场景 1：环境导入失败

**症状：** Cell 3 运行时 `ModuleNotFoundError`（torch、matplotlib 等）

**应对：**
1. 运行 `pip install torch matplotlib tqdm numpy`
2. 确认 `custom_gpt.py` 在 `Custom_GPT_Training/` 目录下
3. 重启 Kernel 后重新运行

### 场景 2：预训练模型不存在

**症状：** Cell 15 输出"预训练模型不存在，创建新模型..."

**应对：**
1. 这不是错误！代码有兜底逻辑，会创建新的随机初始化模型
2. 训练仍然可以进行，但 SFT 效果会差一些（没有预训练基础）
3. 建议先完成 Part 2 预训练再回来做 SFT
4. 如果时间不允许，用随机模型走完流程即可，重点理解原理

### 场景 3：Tokenizer 不存在

**症状：** Cell 10 输出"未找到预训练tokenizer，使用SFT数据构建新词表..."

**应对：**
1. 同样有兜底逻辑，会用 SFT 数据构建新词表（vocab_size=5000, char 模式）
2. 新词表会自动保存到 `models/custom_gpt/tokenizer.pkl`
3. 不影响后续流程

### 场景 4：GPU 显存不足

**症状：** `RuntimeError: CUDA out of memory`

**应对：**
1. 减小 `BATCH_SIZE`（从 8 改为 4）
2. 减小 `MAX_LENGTH`（从 256 改为 128）
3. 切换到 CPU：在 Cell 3 中设置 `device = 'cpu'`（训练时间变长但能跑）

### 场景 5：SFT 数据文件找不到

**症状：** Cell 5 报错 `FileNotFoundError`

**应对：**
1. 确认 `data/` 目录存在且包含 `custom_sft_train.jsonl`、`custom_sft_val.jsonl`、`custom_sft_test.jsonl`
2. 检查 `resolve_data_dir()` 的路径解析逻辑
3. 必要时手动设置 `DATA_DIR` 为你的数据路径

### 场景 6：训练 Loss 不下降

**症状：** Loss 停在初始值附近不动

**应对：**
1. 检查学习率是否正确（应为 5e-5）
2. 检查 labels 是否正确设置（不应全为 -100）
3. 运行 Cell 11 查看 Loss Masking 可视化，确认有非 -100 的 label
4. 确认 `model.train()` 被调用

### 场景 7：CPU 训练时间过长

**症状：** CPU 上 5 个 epoch 训练超过 20 分钟

**应对：**
1. 将 `EPOCHS` 从 5 减少到 2
2. 或减少训练数据：`SFT_DATA = SFT_DATA[:2000]`
3. 提前终止训练，用已有的 loss 趋势说明原理——不影响教学效果

### 场景 8：中文字体不显示

**症状：** Cell 17 的 Matplotlib 图表中中文显示为方框

**应对：**
1. Cell 3 已配置中文字体列表：Microsoft YaHei、SimHei 等
2. 如果仍不显示，安装中文字体或修改 `plt.rcParams["font.sans-serif"]`
3. 不影响训练本身，可跳过图表继续教学